In [ ]:
import subprocess
import sys
import importlib.metadata as md

required = {
    "scikit-learn": "1.6.1",
    "numpy": "2.1.3",
    "pandas": "2.3.3",
    "scipy": "1.16.3",
    "catboost": "1.2.10",
    "interpret-core": "0.7.8",
    "tabicl": "2.2.0",
    "huggingface-hub": "0.36.0",
}

packages = []
for name, version in required.items():
    try:
        installed = md.version(name)
    except md.PackageNotFoundError:
        installed = None
    if installed != version:
        packages.append(f"{name}=={version}")

if packages:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *packages
    ])

print("安装完成。")
print("如果之前已导入numpy/sklearn等，请重启Session，再从第一段运行。")

In [ ]:
from pathlib import Path
import os
import io
import json
import zipfile
import hashlib
import time
import shutil
import platform
import traceback
import importlib.metadata as md

import numpy as np
import pandas as pd
import scipy
import sklearn
import joblib

from scipy.special import logit
from scipy.stats import rankdata
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from threadpoolctl import threadpool_limits
from catboost import CatBoostClassifier
from interpret.glassbox import ExplainableBoostingClassifier
from tabicl import TabICLClassifier
from IPython.display import display, FileLink

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

BOOTSTRAPS = 2000
SEED = 42

NAMES = [
    "LR", "ET", "HGB", "SVM",
    "CatBoost", "EBM", "ensemble", "TabICLv2"
]

STEMS = {
    "features": "bus_v6_features_20261013_120420_94529",
    "training": "bus_v6_external_training_20261013_121259_911015",
    "uclm": "bus_uclm_external_20261013_132503_790776",
}


def sha(data):
    return hashlib.sha256(data).hexdigest()


class Package:
    def __init__(self, path):
        self.path = Path(path)
        self.z = zipfile.ZipFile(path) if self.path.is_file() else None

    def read(self, name):
        if self.z:
            matches = [
                n for n in self.z.namelist()
                if n == name or n.endswith("/" + name)
            ]
            if len(matches) != 1:
                raise ValueError(
                    f"{self.path}: {name} 匹配 {len(matches)} 个文件"
                )
            return self.z.read(matches[0])

        p = self.path / name
        if not p.is_file():
            raise FileNotFoundError(str(p))
        return p.read_bytes()

    def table(self, name):
        return pd.read_csv(
            io.BytesIO(self.read(name)),
            dtype={
                "uid": str,
                "image_id": str,
                "patient_key": str,
                "patient_id": str,
                "cluster_id": str,
            },
        )

    def js(self, name):
        return json.loads(self.read(name))


def discover(stem, anchors):
    # 兼容ZIP与Kaggle自动解压的目录。
    candidates = list(INPUT.rglob(stem + ".zip"))
    candidates += [
        p.parent
        for p in INPUT.rglob(anchors[0])
        if all((p.parent / n).is_file() for n in anchors)
    ]

    groups = {}
    for path in sorted(set(candidates)):
        pkg = Package(path)
        try:
            fingerprint = sha(b"".join(pkg.read(n) for n in anchors))
            groups.setdefault(fingerprint, pkg)
        except (KeyError, ValueError, FileNotFoundError):
            continue

    if len(groups) != 1:
        print("候选：", [str(p) for p in candidates])
        raise RuntimeError(
            f"{stem}: 找到 {len(groups)} 个内容版本。"
            "请只添加指定的三个结果包。"
        )

    pkg = next(iter(groups.values()))
    print("读取：", pkg.path, flush=True)
    return pkg


def joined(meta, features, key):
    if meta[key].isna().any() or features[key].isna().any():
        raise ValueError("存在空ID")
    if not meta[key].is_unique or not features[key].is_unique:
        raise ValueError("存在重复ID")
    if set(meta[key]) != set(features[key]):
        raise ValueError("清单与特征ID不一致")
    return meta.merge(features, on=key, validate="one_to_one")


def predict(bundle, frame, aggregate=False):
    cols = bundle["feature_columns"]

    if bundle["score_type"] == "svm_margin":
        raw = bundle["models"]["SVM"].decision_function(frame[cols])
    else:
        raw = np.mean([
            m.predict_proba(frame[cols])[:, 1]
            for m in bundle["models"].values()
        ], axis=0)

    s = pd.Series(
        raw,
        index=frame.patient_key if aggregate else frame.image_id,
    )

    if aggregate:
        s = s.groupby(level=0).mean().sort_index()

    x = s.to_numpy()
    if bundle["score_type"] != "svm_margin":
        x = logit(np.clip(x, 1e-6, 1 - 1e-6))

    if "calibrator" not in bundle:
        return s, x.reshape(-1, 1)

    p = bundle["calibrator"].predict_proba(x.reshape(-1, 1))[:, 1]
    return pd.Series(p, index=s.index)


def metrics(y, p, threshold):
    positive = y == 1
    n1 = positive.sum()
    n0 = len(y) - n1
    auc = (
        rankdata(p)[positive].sum() - n1 * (n1 + 1) / 2
    ) / (n1 * n0)

    pred = p >= threshold
    return np.array([
        auc,
        np.mean((p - y) ** 2),
        pred[positive].mean(),
        (~pred[~positive]).mean(),
    ])


print("工具准备完成。继续运行第三段。")

In [ ]:
def main():
    names = NAMES.copy()
    failures = []

    if sklearn.__version__ != "1.6.1":
        raise RuntimeError(
            "保存模型要求scikit-learn==1.6.1。"
            "请先运行依赖安装单元并重启Session。"
        )

    f = discover(
        STEMS["features"],
        ["cohort.csv", "features.csv", "v6_extractor.py"],
    )
    t = discover(
        STEMS["training"],
        [
            "source_patient_split.csv",
            "model_hashes.json",
            "models/base_ensemble.joblib",
        ],
    )
    u = discover(
        STEMS["uclm"],
        ["cohort_frozen.csv", "features.csv", "predictions_frozen.csv"],
    )

    protocol = t.js("protocol.json")

    for name, expected in protocol["input_hashes"].items():
        if sha(f.read(name)) != expected:
            raise ValueError(f"V6输入哈希不一致：{name}")

    if sha(u.read("v6_extractor.py")) != sha(f.read("v6_extractor.py")):
        raise ValueError("外部特征提取器版本不一致")

    hashes = t.js("model_hashes.json")
    if hashes != u.js("model_hashes.json"):
        raise ValueError("UCLM与训练包的模型哈希不一致")

    # 恢复原LR及原集成模型。
    bundles = {}
    for name in ["LR", "ensemble"]:
        filename = "base_" + name + ".joblib"
        b = t.read("models/" + filename)
        if sha(b) != hashes[filename]:
            raise ValueError("模型文件哈希不一致")

        bundles[name] = joblib.load(io.BytesIO(b))
        bundles[name]["score_type"] = "probability_logit"

    cols = bundles["LR"]["feature_columns"]

    if len(cols) != 28 or cols != protocol["feature_sets"]["base"]:
        raise ValueError("不是预期的28个基础特征")
    if cols != bundles["ensemble"]["feature_columns"]:
        raise ValueError("两个模型特征列顺序不一致")

    frame = joined(
        f.table("cohort.csv"), f.table("features.csv"), "uid"
    )
    bus = frame.loc[frame.dataset.eq("BUS-BRA")].copy()
    breast = frame.loc[frame.dataset.eq("BrEaST")].copy()
    uclm = joined(
        u.table("cohort_frozen.csv"),
        u.table("features.csv"),
        "image_id",
    )

    for data in [bus, breast, uclm]:
        if data.label.isna().any() or set(data.label.unique()) != {0, 1}:
            raise ValueError("诊断标签异常")

        data["label"] = data.label.astype(int)
        data[cols] = data[cols].apply(pd.to_numeric, errors="raise")

        if np.isinf(data[cols].to_numpy()).any():
            raise ValueError("特征中有无穷值")

    actual = (
        len(bus),
        bus.patient_key.nunique(),
        len(breast),
        len(uclm),
        uclm.cluster_id.nunique(),
    )
    if actual != (1875, 1064, 252, 207, 33):
        raise ValueError(f"队列数量不符合原协议：{actual}")

    for data in [bus, breast]:
        if (
            data.patient_key.isna().any()
            or data.groupby("patient_key").label.nunique().max() != 1
        ):
            raise ValueError("源域或BrEaST患者标签冲突")

    if uclm.cluster_id.isna().any():
        raise ValueError("UCLM缺少聚类ID")

    split = t.table("source_patient_split.csv")
    sy = bus.groupby("patient_key").label.first()

    if (
        not split.patient_key.is_unique
        or set(split.patient_key) != set(sy.index)
    ):
        raise ValueError("患者划分不匹配")

    if not np.array_equal(
        split.label.to_numpy(),
        sy.loc[split.patient_key].to_numpy(),
    ):
        raise ValueError("源域标签与保存划分不匹配")

    roles = {
        role: bus.loc[
            bus.patient_key.isin(
                split.loc[split.role.eq(role), "patient_key"]
            )
        ].copy()
        for role in ["train", "calibration", "threshold"]
    }

    if [x.patient_key.nunique() for x in roles.values()] != [638, 213, 213]:
        raise ValueError("原划分人数不匹配")

    out = WORK / (
        "bus_v8_model_comparison_"
        + time.strftime("%Y%m%d_%H%M%S")
        + "_"
        + str(time.time_ns() % 1000000)
    )
    out.mkdir(parents=True)
    (out / "models").mkdir()
    split.to_csv(out / "source_patient_split.csv", index=False)

    print("数据校验通过。使用相同28特征和原患者划分。", flush=True)

    # 复用原集成模型中的ET、HGB分类器。
    for name in ["ET", "HGB"]:
        bundles[name] = {
            "models": {name: bundles["ensemble"]["models"][name]},
            "feature_columns": cols,
            "score_type": "probability_logit",
        }

    train = roles["train"]
    counts = train.groupby("patient_key").size()
    classes = train.groupby("patient_key").label.first().value_counts()

    weights = (
        1
        / train.patient_key.map(counts).to_numpy()
        / train.label.map(classes).to_numpy()
    )
    weights /= weights.mean()

    # 固定参数SVM，不进行外部队列调参。
    svm = make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True),
        StandardScaler(),
        SVC(
            C=1.0,
            kernel="rbf",
            gamma="scale",
            probability=False,
            cache_size=512,
            random_state=SEED,
        ),
    )
    print("开始拟合：SVM", flush=True)
    with threadpool_limits(limits=2):
        svm.fit(train[cols], train.label, svc__sample_weight=weights)

    bundles["SVM"] = {
        "models": {"SVM": svm},
        "feature_columns": cols,
        "score_type": "svm_margin",
    }

    new_models = {
        "CatBoost": CatBoostClassifier(
            iterations=500,
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            random_seed=42,
            thread_count=2,
            verbose=False,
            allow_writing_files=False,
        ),
        "EBM": ExplainableBoostingClassifier(
            interactions=0,
            outer_bags=1,
            validation_size=0,
            max_rounds=1500,
            learning_rate=0.03,
            n_jobs=2,
            random_state=42,
        ),
        "TabICLv2": TabICLClassifier(
            n_estimators=4,
            batch_size=1,
            device="cpu",
            n_jobs=2,
            use_amp=False,
            random_state=42,
            checkpoint_version="tabicl-classifier-v2-20260212.ckpt",
            verbose=True,
        ),
    }

    audits = []
    for name in names.copy():
        start = time.perf_counter()

        try:
            if name in new_models:
                model = make_pipeline(
                    SimpleImputer(
                        strategy="median", keep_empty_features=True
                    ),
                    new_models[name],
                )
                # TabICLv2官方接口不支持sample_weight。
                fit_args = {} if name == "TabICLv2" else {
                    model.steps[-1][0] + "__sample_weight": weights
                }

                print("开始拟合：", name, flush=True)
                with threadpool_limits(limits=2):
                    model.fit(train[cols], train.label, **fit_args)

                bundles[name] = {
                    "models": {name: model},
                    "feature_columns": cols,
                    "score_type": "probability_logit",
                }

            bundle = bundles[name]

            if name not in ["LR", "ensemble"]:
                raw, x = predict(
                    bundle, roles["calibration"], aggregate=True
                )
                cal = LogisticRegression(
                    C=1000, solver="lbfgs", max_iter=2000
                )
                cal.fit(x, sy.loc[raw.index].to_numpy())
                bundle["calibrator"] = cal

                gate = predict(
                    bundle, roles["threshold"], aggregate=True
                )
                positives = np.sort(
                    gate.loc[sy.loc[gate.index].eq(1)].to_numpy()
                )
                allowed = int(np.floor(.05 * len(positives) + 1e-9))
                bundle["threshold"] = float(positives[allowed])

            else:
                gate = predict(
                    bundle, roles["threshold"], aggregate=True
                )
                old = t.table("source_threshold_predictions.csv")
                old = old.loc[
                    old.feature_set.eq("base") & old.model.eq(name)
                ].set_index("patient_key")

                np.testing.assert_allclose(
                    gate.loc[old.index], old.p,
                    atol=1e-8, rtol=1e-8,
                )
                np.testing.assert_allclose(
                    old.threshold, bundle["threshold"], atol=1e-12
                )

            joblib.dump(
                bundle,
                out / "models" / f"base_{name}.joblib",
                compress=3,
            )

            audits.append({
                "model": name,
                "threshold": bundle["threshold"],
                "source_gate_n": len(gate),
                "source_gate_malignant": int(sy.loc[gate.index].sum()),
                "source_gate_sensitivity": float(
                    (
                        gate.loc[sy.loc[gate.index].eq(1)]
                        >= bundle["threshold"]
                    ).mean()
                ),
                "calibration_slope": float(
                    bundle["calibrator"].coef_[0, 0]
                ),
                "classifier_refitted": name in ["SVM", "CatBoost", "EBM"],
                "weighted_training": name != "TabICLv2",
                "preparation_seconds": time.perf_counter() - start,
            })

            print("已冻结：", name, flush=True)
            pd.DataFrame(audits).to_csv(
                out / "threshold_audit.csv", index=False
            )

        except Exception:
            if name != "TabICLv2":
                raise
            failures.append({
                "model": name,
                "stage": "fit_or_calibration",
                "error": traceback.format_exc(),
            })
            names.remove(name)
            print(
                "TabICLv2运行失败，错误已记录；继续保存其余模型。",
                flush=True,
            )

    # 导出EBM解释性数据：未校准log-odds贡献。
    ebm = bundles["EBM"]["models"]["EBM"].steps[-1][1]
    explanation = ebm.explain_global()

    pd.DataFrame({
        "feature": ebm.term_names_,
        "importance": ebm.term_importances(),
    }).to_csv(out / "ebm_importance.csv", index=False)

    curves = [
        {"feature": feature, "data": explanation.data(i)}
        for i, feature in enumerate(ebm.term_names_)
    ]

    def json_default(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, np.generic):
            return x.item()
        raise TypeError(type(x).__name__)

    (out / "ebm_curves.json").write_text(
        json.dumps(curves, default=json_default)
    )

    # 外部预测前保存全部比较设置。
    spec = {
        "analysis": (
            "Post hoc supplementary model comparison; "
            "external cohorts previously inspected"
        ),
        "features": cols,
        "models": names,
        "requested_models": NAMES,
        "SVM": {
            "C": 1, "kernel": "rbf", "gamma": "scale", "tuning": False
        },
        "calibration": (
            "Source only; LR(C=1000) on patient-mean SVM margins "
            "or clipped probability logits"
        ),
        "threshold": (
            "Source only; highest threshold with >=95% "
            "empirical sensitivity"
        ),
        "reuse": (
            "LR and ensemble retain original calibrators and thresholds; "
            "ET/HGB retain fitted classifiers"
        ),
        "weight_exception": (
            "TabICLv2 uses identical training images without sample "
            "weights; not a weight-matched comparison"
        ),
        "new_model_parameters": {
            k: v.get_params() for k, v in new_models.items()
        },
        "bootstrap": BOOTSTRAPS,
        "bootstrap_seed": 20261014,
        "ci": (
            "Conditional on fitted models, exploratory, "
            "unadjusted multiple comparisons"
        ),
        "unit": "BrEaST case; UCLM image with cluster bootstrap",
        "versions": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": scipy.__version__,
            "sklearn": sklearn.__version__,
            **{
                k: md.version(k)
                for k in [
                    "catboost", "interpret-core", "tabicl",
                    "torch", "huggingface-hub",
                ]
            },
        },
        "sources": {
            k: str(v.path)
            for k, v in [
                ("features", f), ("training", t), ("uclm", u)
            ]
        },
        "original_model_hashes": hashes,
        "input_hashes": {
            key: {n: sha(pkg.read(n)) for n in files}
            for key, pkg, files in [
                ("features", f, ["cohort.csv", "features.csv"]),
                ("training", t, ["source_patient_split.csv"]),
                ("uclm", u, ["cohort_frozen.csv", "features.csv"]),
            ]
        },
    }
    (out / "protocol.json").write_text(
        json.dumps(
            spec, ensure_ascii=False, indent=2, default=json_default
        )
    )

    all_predictions = []
    results = []
    comparisons = []
    checks = []
    fields = ["auc", "brier", "sensitivity", "specificity"]

    for dataset, data, aggregate in [
        ("BrEaST", breast, True),
        ("BUS-UCLM", uclm, False),
    ]:
        print("外部预测及置信区间：", dataset, flush=True)
        series = {}

        for name in names.copy():
            try:
                series[name] = predict(
                    bundles[name], data, aggregate
                )
            except Exception:
                if name != "TabICLv2":
                    raise
                failures.append({
                    "model": name,
                    "stage": dataset,
                    "error": traceback.format_exc(),
                })
                names.remove(name)
                print(
                    "TabICLv2在该外部队列失败，保存其余模型。",
                    flush=True,
                )

        ids = series["LR"].index
        if aggregate:
            y = (
                data.groupby("patient_key").label.first()
                .loc[ids].to_numpy()
            )
            cluster = np.array(ids)
            old = t.table("external_predictions_frozen.csv")
        else:
            index = data.set_index("image_id").loc[ids]
            y = index.label.to_numpy()
            cluster = index.cluster_id.to_numpy()
            old = u.table("predictions_frozen.csv")

        P = np.column_stack([
            series[name].loc[ids].to_numpy() for name in names
        ])
        thresholds = np.array([
            bundles[name]["threshold"] for name in names
        ])

        for j, name in enumerate(names):
            if name in ["LR", "ensemble"]:
                if aggregate:
                    ref = old.loc[
                        old.feature_set.eq("base") & old.model.eq(name)
                    ].set_index("patient_key")["p"]
                else:
                    ref = old.loc[
                        old.configuration.eq("base_" + name)
                    ].set_index("image_id")["probability"]

                if set(ref.index) != set(ids):
                    raise ValueError("旧预测ID不匹配")

                delta = float(
                    np.max(np.abs(P[:, j] - ref.loc[ids].to_numpy()))
                )
                np.testing.assert_allclose(
                    P[:, j], ref.loc[ids], atol=1e-8, rtol=1e-8
                )
                checks.append({
                    "dataset": dataset,
                    "model": name,
                    "max_abs_difference": delta,
                })

            all_predictions.append(pd.DataFrame({
                "dataset": dataset,
                "id": ids,
                "cluster_id": cluster,
                "label": y,
                "model": name,
                "p": P[:, j],
                "threshold": thresholds[j],
                "prediction": (P[:, j] >= thresholds[j]).astype(int),
            }))

        point = np.stack([
            metrics(y, P[:, j], thresholds[j])
            for j in range(len(names))
        ])

        rng = np.random.default_rng(20261014)
        groups = [
            np.flatnonzero(cluster == c) for c in np.unique(cluster)
        ]
        strata = [np.flatnonzero(y == v) for v in [0, 1]]
        samples = []
        attempts = 0

        while (
            len(samples) < BOOTSTRAPS
            and attempts < BOOTSTRAPS * 10
        ):
            attempts += 1

            if aggregate:
                ix = np.concatenate([
                    rng.choice(g, len(g), replace=True)
                    for g in strata
                ])
            else:
                ix = np.concatenate([
                    groups[j]
                    for j in rng.integers(
                        0, len(groups), len(groups)
                    )
                ])

            if len(np.unique(y[ix])) < 2:
                continue

            samples.append(np.stack([
                metrics(y[ix], P[ix, j], thresholds[j])
                for j in range(len(names))
            ]))

        if len(samples) != BOOTSTRAPS:
            raise RuntimeError("有效bootstrap数量不足")

        samples = np.stack(samples)

        for j, name in enumerate(names):
            row = {
                "dataset": dataset,
                "model": name,
                "n": len(y),
                "clusters": len(groups),
                "threshold": thresholds[j],
                "missed_malignant": int(
                    ((P[:, j] < thresholds[j]) & (y == 1)).sum()
                ),
                "false_positive": int(
                    ((P[:, j] >= thresholds[j]) & (y == 0)).sum()
                ),
            }

            for k, metric in enumerate(fields):
                lo, hi = np.quantile(
                    samples[:, j, k], [.025, .975]
                )
                row.update({
                    metric: point[j, k],
                    metric + "_low": lo,
                    metric + "_high": hi,
                })

                if j:
                    dlo, dhi = np.quantile(
                        samples[:, j, k] - samples[:, 0, k],
                        [.025, .975],
                    )
                    comparisons.append({
                        "dataset": dataset,
                        "comparison": name + " minus LR",
                        "metric": metric,
                        "difference": point[j, k] - point[0, k],
                        "low": dlo,
                        "high": dhi,
                        "note": (
                            "Negative Brier favors comparator; "
                            "other positive differences favor comparator. "
                            "Exploratory, unadjusted."
                        ),
                    })

            results.append(row)

        # 每个队列完成后立即保存。
        pd.concat(all_predictions).to_csv(
            out / "predictions.csv", index=False
        )
        pd.DataFrame(results).to_csv(
            out / "metrics.csv", index=False
        )
        pd.DataFrame(comparisons).to_csv(
            out / "paired_differences_vs_LR.csv", index=False
        )

    (out / "failures.json").write_text(
        json.dumps(failures, ensure_ascii=False, indent=2)
    )
    (out / "completion.json").write_text(
        json.dumps({
            "requested": NAMES,
            "failed": failures,
            "complete": not failures,
        }, ensure_ascii=False, indent=2)
    )

    pd.DataFrame(checks).to_csv(
        out / "original_prediction_reproduction.csv", index=False
    )
    (out / "model_hashes.json").write_text(
        json.dumps({
            p.name: sha(p.read_bytes())
            for p in (out / "models").glob("*.joblib")
        }, indent=2)
    )

    try:
        source = Path(__file__).read_text()
    except NameError:
        source = "\n\n".join(
            get_ipython().history_manager.input_hist_raw[1:]
        )
    (out / "executed_code.py").write_text(source)

    (out / "README.txt").write_text(
        "V8事后补充比较，不替换V6主分析。"
        "28个相同基础特征和相同源域患者划分。"
        "LR/ensemble复用原模型、校准和阈值；ET/HGB复用分类器。"
        "SVM/CatBoost/EBM新训练。"
        "TabICLv2预训练模型CPU上下文推断，4个成员，"
        "不支持sample_weight，使用相同训练图像但未加权。"
        "此差异必须在论文披露。"
        "所有新增校准和阈值只用BUS-BRA。"
        "外部结果已看过，属事后分析。"
        "EBM曲线为未校准模型log-odds贡献，不是因果效应。"
        "ET/HGB/LR/ensemble未重新训练，"
        "因此preparation_seconds不能当作总训练时间比较。"
        "CI条件于已训练模型，未作多重比较校正。"
        "检查completion.json确认8模型是否全部完成。"
    )

    archive = shutil.make_archive(str(out), "zip", out)

    display(pd.DataFrame(results)[[
        "dataset", "model", "auc", "auc_low", "auc_high",
        "brier", "sensitivity", "specificity",
        "missed_malignant", "false_positive",
    ]].round(4))

    if not failures:
        print("全部8模型完成。")
    else:
        print("部分完成，请查看failures.json；不能当成8模型全部完成。")

    print("请下载并发回：", archive)
    display(FileLink(archive))
    return out


main()

# ================= END OF CODE =================

In [ ]:
from pathlib import Path
import io
import json
import zipfile
import hashlib
import time
import shutil

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from IPython.display import display, FileLink

INPUT = Path("/kaggle/working")
WORK = Path("/kaggle/working")

STEM = "bus_v8_model_comparison_20261014_141357_851028"
MODELS = [
    "LR", "ET", "HGB", "SVM",
    "CatBoost", "EBM", "ensemble", "TabICLv2"
]

# 固定探索范围，不根据外部结果选择最优阈值。
# 这是分析网格，不是临床推荐阈值范围。
THRESHOLDS = np.arange(2, 51) / 100
B = 2000

# ---------- 1. 查找V8结果 ----------
packages = {}

candidates = list(INPUT.rglob(STEM + ".zip"))
candidates += [
    p.parent
    for p in INPUT.rglob("predictions.csv")
    if (p.parent / "completion.json").is_file()
    and (p.parent / "ebm_curves.json").is_file()
]

for path in candidates:
    def read(name, path=path):
        if path.is_dir():
            return (path / name).read_bytes()

        with zipfile.ZipFile(path) as z:
            matches = [
                n for n in z.namelist()
                if n == name or n.endswith("/" + name)
            ]
            if len(matches) != 1:
                raise ValueError(f"{name}: 文件数量异常")
            return z.read(matches[0])

    content = read("predictions.csv")
    fingerprint = hashlib.sha256(content).hexdigest()
    packages.setdefault(fingerprint, (path, read))

if len(packages) != 1:
    raise RuntimeError(
        "请仅添加指定的完整V8结果ZIP；当前找到内容版本数："
        + str(len(packages))
    )

source, read = next(iter(packages.values()))

if not json.loads(read("completion.json"))["complete"]:
    raise ValueError("V8没有全部完成")

df = pd.read_csv(
    io.BytesIO(read("predictions.csv")),
    dtype={"id": str, "cluster_id": str},
    float_precision="round_trip",
)

if df.duplicated(["dataset", "model", "id"]).any():
    raise ValueError("重复ID")

required = ["id", "cluster_id", "label", "p", "threshold", "prediction"]
if df[required].isna().any().any():
    raise ValueError("存在缺失值")

if not df.p.between(0, 1).all() or set(df.label.unique()) != {0, 1}:
    raise ValueError("概率或标签异常")

if set(df.dataset) != {"BrEaST", "BUS-UCLM"}:
    raise ValueError("队列异常")

out = WORK / (
    "bus_v9_decision_analysis_" + time.strftime("%Y%m%d_%H%M%S")
)
out.mkdir(parents=True, exist_ok=False)

protocol = {
    "analysis": (
        "Post hoc exploratory decision analysis; "
        "no model fitting or threshold optimization"
    ),
    "threshold_grid": THRESHOLDS.tolist(),
    "bootstrap": B,
    "seed": 20261015,
    "prevalence": (
        "Observed selected-cohort prevalence, "
        "not target clinical prevalence"
    ),
    "units": (
        "BrEaST case; BUS-UCLM image with patient-cluster bootstrap"
    ),
    "ci": (
        "Pointwise 95% percentile intervals; "
        "not simultaneous; no multiplicity correction"
    ),
    "interpretation": (
        "Not evidence of achieved biopsy reduction "
        "or prospective clinical utility"
    ),
    "source": str(source),
    "prediction_sha256": hashlib.sha256(
        read("predictions.csv")
    ).hexdigest(),
}
(out / "protocol.json").write_text(json.dumps(protocol, indent=2))

# ---------- 2. 保留阈值审计问题 ----------
audit = pd.read_csv(io.BytesIO(read("threshold_audit.csv")))
audit["below_95_reported"] = audit.source_gate_sensitivity < .95
audit.to_csv(out / "source_gate_audit_UNRESOLVED.csv", index=False)

print("源域阈值审计中需核查的记录：")
display(audit.loc[audit.below_95_reported])

# 仅有V8包无法重建源域阈值边界的数值行为，不修改阈值。
near = df.loc[np.abs(df.p - df.threshold) <= 1e-12].copy()
near.to_csv(out / "external_near_threshold.csv", index=False)

if not np.array_equal(
    (df.p >= df.threshold).astype(int),
    df.prediction.astype(int),
):
    raise ValueError(
        "保存预测与阈值不一致，请先核查，不能自动改标签"
    )

# ---------- 3. 决策曲线及配对bootstrap ----------
rows = []
differences = []
counts = []
summaries = []

for dataset in ["BrEaST", "BUS-UCLM"]:
    g = df.loc[df.dataset.eq(dataset)]

    if set(g.model) != set(MODELS):
        raise ValueError("缺少模型")

    ref = (
        g.loc[g.model.eq("LR")]
        .set_index("id")
        .sort_index()
    )
    ids = ref.index
    y = ref.label.to_numpy(dtype=int)
    clusters = ref.cluster_id.to_numpy()

    arrays = []
    for model in MODELS:
        q = g.loc[g.model.eq(model)].set_index("id")
        if set(q.index) != set(ids):
            raise ValueError("模型ID不一致")

        q = q.loc[ids]
        if (
            not np.array_equal(q.label, y)
            or not np.array_equal(q.cluster_id, clusters)
        ):
            raise ValueError("模型间标签或患者分组不一致")

        arrays.append(q.p.to_numpy())

    P = np.column_stack(arrays)

    # 每个病例 × 模型 × 阈值
    decisions = P[:, :, None] >= THRESHOLDS[None, None, :]
    tp = decisions * (y[:, None, None] == 1)
    fp = decisions * (y[:, None, None] == 0)

    odds = THRESHOLDS / (1 - THRESHOLDS)
    utility = tp - fp * odds[None, None, :]
    point = utility.mean(axis=0)
    all_nb = y.mean() - (1 - y.mean()) * odds

    rng = np.random.default_rng(20261015)
    group_ids = np.unique(clusters)
    groups = [
        np.flatnonzero(clusters == c)
        for c in group_ids
    ]
    strata = [
        np.flatnonzero(y == v)
        for v in [0, 1]
    ]

    # 预先按患者汇总，加快UCLM聚类bootstrap。
    sums = np.stack([
        utility[ix].sum(axis=0)
        for ix in groups
    ])
    sizes = np.array([len(ix) for ix in groups])
    positives = np.array([y[ix].sum() for ix in groups])

    boot = []
    boot_all = []

    for b in range(B):
        if dataset == "BrEaST":
            ix = np.concatenate([
                rng.choice(s, len(s), replace=True)
                for s in strata
            ])
            nb = utility[ix].mean(axis=0)
            prev = y[ix].mean()
        else:
            ix = rng.integers(0, len(groups), len(groups))
            nb = sums[ix].sum(axis=0) / sizes[ix].sum()
            prev = positives[ix].sum() / sizes[ix].sum()

        boot.append(nb)
        boot_all.append(prev - (1 - prev) * odds)

    boot = np.stack(boot)
    boot_all = np.stack(boot_all)

    fig, ax = plt.subplots(figsize=(9, 6))

    for j, model in enumerate(MODELS):
        lo, hi = np.quantile(
            boot[:, j, :], [.025, .975], axis=0
        )
        dlo, dhi = np.quantile(
            boot[:, j, :] - boot[:, 0, :],
            [.025, .975], axis=0,
        )
        alo, ahi = np.quantile(
            boot[:, j, :] - boot_all,
            [.025, .975], axis=0,
        )

        ax.plot(
            THRESHOLDS, point[j],
            label=model,
            lw=2 if model == "LR" else 1.2,
        )

        for k, threshold in enumerate(THRESHOLDS):
            rows.append({
                "dataset": dataset,
                "model": model,
                "threshold": threshold,
                "net_benefit": point[j, k],
                "low": lo[k],
                "high": hi[k],
                "all_positive": all_nb[k],
                "none_positive": 0,
                "delta_vs_all": point[j, k] - all_nb[k],
                "delta_vs_all_low": alo[k],
                "delta_vs_all_high": ahi[k],
            })

            differences.append({
                "dataset": dataset,
                "model": model,
                "threshold": threshold,
                "delta_vs_LR": point[j, k] - point[0, k],
                "low": dlo[k],
                "high": dhi[k],
            })

            counts.append({
                "dataset": dataset,
                "model": model,
                "threshold": threshold,
                "n": len(y),
                "TP": int(tp[:, j, k].sum()),
                "FP": int(fp[:, j, k].sum()),
                "FN": int(y.sum() - tp[:, j, k].sum()),
                "TN": int((1 - y).sum() - fp[:, j, k].sum()),
            })

    ax.plot(
        THRESHOLDS, all_nb, "k--", label="All positive"
    )
    ax.axhline(
        0, color="gray", ls=":", label="None positive"
    )
    title = dataset + (
        " (case level)"
        if dataset == "BrEaST"
        else " (image level; selected cohort)"
    )
    ax.set(
        xlabel="Exploratory threshold probability",
        ylabel="Net benefit",
        title=title,
    )
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(out / (dataset + "_DCA.png"), dpi=300)
    fig.savefig(out / (dataset + "_DCA.pdf"))
    plt.close(fig)

    # ---------- 4. 原冻结阈值下的LR/TabICLv2分歧病例 ----------
    d = ref[
        ["cluster_id", "label", "p", "threshold", "prediction"]
    ].rename(columns={
        "p": "LR_p",
        "threshold": "LR_threshold",
        "prediction": "LR_prediction",
    })

    q = (
        g.loc[g.model.eq("TabICLv2")]
        .set_index("id")
        .loc[ids]
    )
    d["TabICLv2_p"] = q.p
    d["TabICLv2_threshold"] = q.threshold
    d["TabICLv2_prediction"] = q.prediction
    d["LR_correct"] = d.LR_prediction.eq(d.label)
    d["TabICLv2_correct"] = d.TabICLv2_prediction.eq(d.label)
    d["disagree"] = d.LR_prediction.ne(d.TabICLv2_prediction)

    d.to_csv(out / (dataset + "_LR_TabICLv2_cases.csv"))

    for label in [0, 1]:
        a = d.loc[d.label.eq(label)]
        summaries.append({
            "dataset": dataset,
            "label": label,
            "n": len(a),
            "both_correct": int(
                (a.LR_correct & a.TabICLv2_correct).sum()
            ),
            "only_LR_correct": int(
                (a.LR_correct & ~a.TabICLv2_correct).sum()
            ),
            "only_TabICLv2_correct": int(
                (~a.LR_correct & a.TabICLv2_correct).sum()
            ),
            "both_wrong": int(
                (~a.LR_correct & ~a.TabICLv2_correct).sum()
            ),
        })

    print(dataset, "完成。")

# ---------- 5. 保存 ----------
pd.DataFrame(rows).to_csv(
    out / "decision_curves.csv", index=False
)
pd.DataFrame(differences).to_csv(
    out / "paired_net_benefit_vs_LR.csv", index=False
)
pd.DataFrame(counts).to_csv(
    out / "threshold_confusion_counts.csv", index=False
)
pd.DataFrame(summaries).to_csv(
    out / "disagreement_summary.csv", index=False
)

(out / "README.txt").write_text(
    "事后探索分析，无重新训练、重校准或选择最优阈值。"
    "0.02至0.50是探索网格，不是临床推荐范围。"
    "采用所选队列观察患病比例，不能外推真实临床人群。"
    "BUS-UCLM为图像单位，不可解释成患者活检减少。"
    "置信区间逐点、未多重校正，不可挑区间宣称整体优越。"
    "源域ensemble低于95%的记录保留待核查；"
    "V8包不足以确定其数值原因，本代码不修正阈值。"
)

try:
    code = Path(__file__).read_text()
except NameError:
    code = "\n\n".join(
        get_ipython().history_manager.input_hist_raw[1:]
    )
(out / "executed_code.py").write_text(code)

archive = shutil.make_archive(str(out), "zip", out)

display(pd.DataFrame(summaries))
print("完成，请下载：", archive)
display(FileLink(archive))

# ================= END OF CODE =================

In [ ]:
from pathlib import Path
import io
import json
import zipfile
import hashlib
import time
import shutil

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]
WORK = Path("/kaggle/working")
B = 2000
STEM = "bus_v6_features_20261013_120420_94529"

# 名称、临床字段、正组、负组、影像特征、缺失标志。
# 固定方向：预期正组的特征值更高；不根据结果翻转方向。
SPEC = [
    (
        "Shape_irregular", "Shape", "irregular",
        ["oval", "round"], "shape__roughness", "none"
    ),
    (
        "Shape_oval", "Shape", "oval",
        ["round"], "shape__elongation", "none"
    ),
    (
        "Margin_non_circumscribed", "Margin", "not circumscribed",
        ["circumscribed"], "shape__roughness", "none"
    ),
    (
        "Echo_heterogeneous", "Echogenicity", "heterogeneous",
        ["hypoechoic"], "core__std", "core__missing"
    ),
    (
        "Echo_hypoechoic", "Echogenicity", "hypoechoic",
        ["anechoic"], "core__median", "core__missing"
    ),
    (
        "Posterior_enhancement", "Posterior_features", "enhancement",
        ["shadowing"], "postdepth__delta", "postdepth__missing"
    ),
]

# ---------- 1. 查找V6特征包 ----------
packages = {}

for root in ROOTS:
    if not root.exists():
        continue

    paths = list(root.rglob(STEM + ".zip"))
    paths += [
        p.parent
        for p in root.rglob("cohort.csv")
        if (p.parent / "features.csv").is_file()
        and (p.parent / "v6_extractor.py").is_file()
    ]

    for path in paths:
        def read(name, path=path):
            if path.is_dir():
                return (path / name).read_bytes()

            with zipfile.ZipFile(path) as z:
                matches = [
                    n for n in z.namelist()
                    if n == name or n.endswith("/" + name)
                ]
                if len(matches) != 1:
                    raise ValueError("文件不唯一：" + name)
                return z.read(matches[0])

        fingerprint = hashlib.sha256(
            read("cohort.csv") + read("features.csv")
        ).hexdigest()
        packages.setdefault(fingerprint, (path, read))

if len(packages) != 1:
    raise RuntimeError(
        "请保留一份完整V6 features结果；发现内容版本数="
        + str(len(packages))
    )

source, read = next(iter(packages.values()))
print("特征来源：", source)

# ---------- 2. 查找官方临床表 ----------
required = {
    "CaseID", "Image_filename", "Shape",
    "Margin", "Echogenicity", "Posterior_features",
}
tables = {}

for root in ROOTS:
    if not root.exists():
        continue

    for p in root.rglob("*.xlsx"):
        if "clinical" not in p.name.lower():
            continue

        book = pd.read_excel(p, sheet_name=None)
        for sheet, table in book.items():
            if required.issubset(table.columns):
                key = hashlib.sha256(p.read_bytes()).hexdigest() + str(sheet)
                tables.setdefault(key, (p, sheet, table))

if len(tables) != 1:
    raise RuntimeError(
        "需要一份BrEaST官方临床XLSX；当前匹配数="
        + str(len(tables))
    )

clinical_path, sheet, clinical = next(iter(tables.values()))
print("临床表：", clinical_path)

# ---------- 3. 严格按ID和图像文件名核对 ----------
meta = pd.read_csv(
    io.BytesIO(read("cohort.csv")),
    dtype={"uid": str, "image_id": str},
)
features = pd.read_csv(
    io.BytesIO(read("features.csv")),
    dtype={"uid": str},
)

if (
    not meta.uid.is_unique
    or not features.uid.is_unique
    or set(meta.uid) != set(features.uid)
):
    raise ValueError("清单与特征ID异常")

m = meta.loc[meta.dataset.eq("BrEaST")].copy()
m["CaseID"] = pd.to_numeric(m.image_id, errors="raise").astype(int)
clinical["CaseID"] = pd.to_numeric(
    clinical.CaseID, errors="raise"
).astype(int)

if not clinical.CaseID.is_unique or not m.CaseID.is_unique:
    raise ValueError("重复CaseID")

if not set(m.CaseID).issubset(clinical.CaseID):
    raise ValueError("临床ID缺失")

d = (
    m.merge(features, on="uid", validate="one_to_one")
    .merge(clinical, on="CaseID", validate="one_to_one")
)

if len(d) != 252:
    raise ValueError("预期252个BrEaST病灶病例")

if not all(
    Path(str(a)).name == str(b)
    for a, b in zip(d.image_path, d.Image_filename)
):
    raise ValueError("CaseID与图像文件名不一致")

for col in ["Shape", "Margin", "Echogenicity", "Posterior_features"]:
    d[col] = d[col].fillna("").astype(str).str.strip().str.lower()

# ---------- 4. 分析前保存规则 ----------
out = WORK / (
    "breast_v10_semantics_" + time.strftime("%Y%m%d_%H%M%S")
)
out.mkdir(parents=True, exist_ok=False)

protocol = {
    "status": (
        "Post hoc exploratory semantic association, "
        "not cancer classification"
    ),
    "contrasts": SPEC,
    "direction": (
        "Larger feature expected in positive group; "
        "never reverse after results"
    ),
    "tests": (
        "Two-sided Mann-Whitney; "
        "Holm correction across six contrasts"
    ),
    "effect": (
        "Directional association AUC="
        "P(Xpositive>Xnegative)+0.5*P(tie)"
    ),
    "bootstrap": B,
    "seed": 42,
    "missing": "Complete measurable cases per contrast, no imputation",
    "limitations": [
        "Shared image and manual tumor masks; "
        "not independent mechanistic validation",
        "Brightness affected by acquisition; no causal interpretation",
        "Posterior availability may select cases; "
        "combined posterior signs excluded",
        "No selection by significance; retain all six comparisons",
    ],
    "features_source": str(source),
    "clinical_source": str(clinical_path),
    "sheet": sheet,
    "feature_sha256": hashlib.sha256(
        read("features.csv")
    ).hexdigest(),
    "clinical_sha256": hashlib.sha256(
        clinical_path.read_bytes()
    ).hexdigest(),
}
(out / "protocol.json").write_text(
    json.dumps(protocol, ensure_ascii=False, indent=2)
)

# ---------- 5. 六项固定检验 ----------
rows = []
availability = []
cases = []
strata = []

fig, axes = plt.subplots(
    2, 3, figsize=(15, 9), layout="constrained"
)
rng = np.random.default_rng(42)


def effect(a, b):
    result = mannwhitneyu(
        a, b, alternative="two-sided", method="asymptotic"
    )
    return float(result.statistic / (len(a) * len(b)))


for ax, specification in zip(axes.flat, SPEC):
    name, col, positive, negative, feature, flag = specification

    if name == "Margin_non_circumscribed":
        pos = d[col].str.startswith("not circumscribed")
    else:
        pos = d[col].eq(positive)

    neg = d[col].isin(negative)
    values = pd.to_numeric(d[feature], errors="raise")
    usable = np.isfinite(values)

    if flag != "none":
        usable &= pd.to_numeric(
            d[flag], errors="raise"
        ).eq(0)

    for title, mask in [("positive", pos), ("negative", neg)]:
        availability.append({
            "contrast": name,
            "group": title,
            "eligible": int(mask.sum()),
            "measured": int((mask & usable).sum()),
            "missing": int((mask & ~usable).sum()),
        })

    g = d.loc[
        (pos | neg) & usable,
        ["CaseID", "label", col],
    ].copy()
    g["feature"] = feature
    g["value"] = values.loc[g.index]
    g["positive"] = pos.loc[g.index].astype(int)
    g["contrast"] = name
    cases.append(g)

    a = values[pos & usable].to_numpy()
    b = values[neg & usable].to_numpy()

    if min(len(a), len(b)) < 3:
        rows.append({
            "contrast": name,
            "feature": feature,
            "n_positive": len(a),
            "n_negative": len(b),
            "p": 1.0,
            "status": "insufficient",
        })
        ax.set_title(name + " (insufficient)")
        continue

    test = mannwhitneyu(
        a, b, alternative="two-sided", method="asymptotic"
    )
    auc = effect(a, b)

    boot = np.array([
        effect(
            rng.choice(a, len(a), replace=True),
            rng.choice(b, len(b), replace=True),
        )
        for _ in range(B)
    ])
    lo, hi = np.quantile(boot, [.025, .975])

    rows.append({
        "contrast": name,
        "feature": feature,
        "n_positive": len(a),
        "n_negative": len(b),
        "positive_median": np.median(a),
        "negative_median": np.median(b),
        "association_auc": auc,
        "auc_low": lo,
        "auc_high": hi,
        "rank_biserial": 2 * auc - 1,
        "p": float(test.pvalue),
        "status": "ok",
    })

    # 同时按良恶性分层作描述性检查。
    # 不训练模型，也不将其解释成因果调整。
    for label in [0, 1]:
        aa = g.loc[
            g.label.eq(label) & g.positive.eq(1), "value"
        ].to_numpy()
        bb = g.loc[
            g.label.eq(label) & g.positive.eq(0), "value"
        ].to_numpy()

        strata.append({
            "contrast": name,
            "diagnosis_label": label,
            "n_positive": len(aa),
            "n_negative": len(bb),
            "association_auc": (
                effect(aa, bb)
                if min(len(aa), len(bb)) >= 3
                else np.nan
            ),
            "note": (
                "Descriptive within-diagnosis check, "
                "not adjusted causal estimate"
            ),
        })

    ax.boxplot([b, a], showfliers=False)
    ax.set_xticks([1, 2], ["Negative", "Positive"])

    for x, v in [(1, b), (2, a)]:
        ax.scatter(
            x + rng.uniform(-.12, .12, len(v)),
            v, s=9, alpha=.35,
        )

    ax.set_title(
        name + f"\nAssociation AUC={auc:.3f} [{lo:.3f}, {hi:.3f}]",
        fontsize=10,
    )
    ax.set_ylabel(feature)
    print(name, "完成。", flush=True)

# ---------- 6. Holm校正及结果保存 ----------
res = pd.DataFrame(rows)

order = np.argsort(res.p.to_numpy())
adjusted = np.minimum(
    1,
    np.maximum.accumulate(
        (len(res) - np.arange(len(res)))
        * res.p.to_numpy()[order]
    ),
)
res["p_holm"] = np.nan
res.loc[res.index[order], "p_holm"] = adjusted

res.to_csv(out / "semantic_associations.csv", index=False)

pd.DataFrame(availability).to_csv(
    out / "measurement_availability.csv", index=False
)
pd.DataFrame(strata).to_csv(
    out / "within_diagnosis_descriptive.csv", index=False
)
pd.concat(cases, ignore_index=True).to_csv(
    out / "case_level_measurements.csv", index=False
)

fig.savefig(out / "semantic_associations.png", dpi=250)
fig.savefig(out / "semantic_associations.pdf")
plt.close(fig)

(out / "README.txt").write_text(
    "这里AUC衡量影像特征与医生描述的对应关系，"
    "不是良恶性诊断AUC。方向事先固定，"
    "不将低于0.5的结果翻转。"
    "6项双侧检验使用Holm校正。"
    "置信区间为未校正逐项区间。"
    "每项有效样本和缺失比例单独报告。"
    "形状指标来自人工轮廓，"
    "与医生图像描述并非独立证据。"
    "分层结果仅描述，不能证明消除混杂。"
)

try:
    code = Path(__file__).read_text()
except NameError:
    code = "\n\n".join(
        get_ipython().history_manager.input_hist_raw[1:]
    )
(out / "executed_code.py").write_text(code)

archive = shutil.make_archive(str(out), "zip", out)

display(res.round(4))
print("完成，请下载：", archive)
display(FileLink(archive))

# ================= END OF CODE =================

In [ ]:
# ==================== 开始 ====================
from pathlib import Path
from datetime import datetime
import io
import json
import zipfile
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, FileLink

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

# 自动发现失败时填写上传后的 ZIP 或解压目录路径。
SOURCE_PATH = ""

OUT = WORK / (
    "bus_v11_figures_" + datetime.now().strftime("%Y%m%d_%H%M%S")
)
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

ANCHOR = "global_feature_summary.csv"

# ---------- 1. 找到结果包 ----------
if SOURCE_PATH.strip():
    source = Path(SOURCE_PATH.strip())
else:
    directories = sorted({
        p.parent for p in INPUT.rglob(ANCHOR)
        if (p.parent / "validation.json").exists()
    })

    archives = []
    if not directories:
        for p in INPUT.rglob("*.zip"):
            try:
                with zipfile.ZipFile(p) as z:
                    names = {Path(n).name for n in z.namelist()}
                    if {ANCHOR, "validation.json"}.issubset(names):
                        archives.append(p)
            except zipfile.BadZipFile:
                pass

    candidates = directories if directories else archives
    if len(candidates) != 1:
        print("候选位置：")
        for p in candidates:
            print(p)
        raise RuntimeError(
            "请添加 bus_v11_interpretability.zip；"
            "若有多个版本，请在 SOURCE_PATH 填写明确路径。"
        )
    source = candidates[0]

print("读取：", source)

def read_bytes(name):
    if source.is_dir():
        direct = source / name
        candidates = (
            [direct] if direct.is_file()
            else list(source.rglob(name))
        )
        if len(candidates) != 1:
            raise RuntimeError(
                f"{name} 匹配到 {len(candidates)} 个文件"
            )
        return candidates[0].read_bytes()

    with zipfile.ZipFile(source) as z:
        matches = [
            n for n in z.namelist()
            if not n.endswith("/")
            and (n == name or n.endswith("/" + name))
        ]
        if len(matches) != 1:
            raise RuntimeError(
                f"{name} 匹配到 {len(matches)} 个文件"
            )
        return z.read(matches[0])

def read_csv(name):
    return pd.read_csv(
        io.BytesIO(read_bytes(name)),
        dtype={"id": str, "cluster_id": str},
        float_precision="round_trip",
    )

validation = json.loads(read_bytes("validation.json"))
protocol = json.loads(read_bytes("protocol.json"))
summary = read_csv(ANCHOR)

print("原始预测还原检查：")
display(pd.DataFrame(validation))

if any(
    x["max_abs_prediction_error"] >= 1e-8
    for x in validation
):
    raise RuntimeError("贡献未能准确还原预测，停止绘图。")

# 保留追溯信息
for filename in [
    "protocol.json",
    "validation.json",
    "global_feature_summary.csv",
    "selected_case_provenance.csv",
]:
    (OUT / filename).write_bytes(read_bytes(filename))

# 使用准确名称，避免把 roughness 过度解释成边缘毛刺。
DISPLAY_NAMES = {
    "shape__roughness": "Inverse circularity",
    "shape__elongation": "Elongation",
    "shape__aspect": "Aspect ratio",
    "rim__contrast": "Boundary contrast",
    "rim__std": "Boundary intensity SD",
    "core__p10": "Core intensity: 10th percentile",
    "core__median": "Core median intensity",
    "core__std": "Core intensity SD",
    "core__edge": "Core edge feature",
    "core__contrast": "Core contrast",
}

def pretty(name):
    return DISPLAY_NAMES.get(name, name.replace("__", ": "))

def save_figure(fig, name):
    fig.savefig(OUT / f"{name}.png", dpi=300,
                bbox_inches="tight")
    fig.savefig(OUT / f"{name}.pdf",
                bbox_inches="tight")
    plt.show()
    plt.close(fig)

# ---------- 2. 两个外部队列的全局贡献 ----------
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, dataset in zip(axes, ["BrEaST", "BUS-UCLM"]):
    top = (
        summary[summary["dataset"].eq(dataset)]
        .nlargest(10, "cluster_equal_mean_abs")
        .sort_values("cluster_equal_mean_abs")
    )

    ax.barh(
        [pretty(x) for x in top["feature"]],
        top["cluster_equal_mean_abs"],
        color="#287da5",
    )
    ax.set_title(dataset + " — frozen LR")
    ax.set_xlabel(
        "Mean absolute contribution\n"
        "(calibrated log-odds; equal cluster weight)"
    )
    ax.grid(axis="x", alpha=0.15)
    ax.set_axisbelow(True)

fig.tight_layout()
save_figure(fig, "Figure_global_contributions")

# ---------- 3. 全队列特征贡献热图 ----------
# 每行是一例/一幅图像；每列是一项特征。
# 红色：相对训练均值提高模型恶性概率。
# 蓝色：相对训练均值降低模型恶性概率。
# 排序仅用于可视化，不用于训练或改变阈值。
for dataset in ["BrEaST", "BUS-UCLM"]:
    contributions = read_csv(
        dataset + "_all_feature_contributions.csv"
    ).set_index("id")

    details = read_csv(
        dataset + "_all_group_contributions.csv"
    ).set_index("id")

    if not contributions.index.is_unique:
        raise ValueError("贡献表存在重复 ID。")
    if not details.index.is_unique:
        raise ValueError("预测表存在重复 ID。")
    if set(contributions.index) != set(details.index):
        raise ValueError("贡献表与预测表 ID 不一致。")

    ordered = (
        details.reset_index()
        .sort_values(["p", "id"])
        .set_index("id")
    )

    features = (
        summary[summary["dataset"].eq(dataset)]
        .nlargest(10, "cluster_equal_mean_abs")["feature"]
        .tolist()
    )

    matrix = contributions.loc[
        ordered.index, features
    ].to_numpy(dtype=float)

    # 使用实际最大绝对值，避免颜色截断隐藏极端贡献。
    vmax = max(float(np.max(np.abs(matrix))), 1e-6)

    fig, ax = plt.subplots(figsize=(11, 7))
    im = ax.imshow(
        matrix,
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        aspect="auto",
        interpolation="nearest",
    )

    ax.set_xticks(range(len(features)))
    ax.set_xticklabels(
        [pretty(x) for x in features],
        rotation=45, ha="right",
    )
    ax.set_ylabel(
        "Cases/images sorted by increasing LR probability"
    )
    ax.set_title(
        dataset + " — signed LR feature contributions\n"
        "Feature-level explanation, not pixel saliency"
    )

    colorbar = fig.colorbar(im, ax=ax, pad=0.02)
    colorbar.set_label("Calibrated log-odds contribution")

    fig.tight_layout()
    save_figure(fig, dataset + "_feature_heatmap")

    # 输出对应排序和完整贡献，便于追溯。
    ordered.to_csv(OUT / (dataset + "_heatmap_case_order.csv"))
    contributions.to_csv(
        OUT / (dataset + "_all_feature_contributions.csv")
    )

# ---------- 4. 显示原图＋轮廓＋病例贡献 ----------
# 以下为已经完成图像哈希核对、特征重新提取核对的病例图。
# 每队列6例：TN、TP、FP、FN和两个方向的模型分歧。
# 并非随机挑选“效果好”的病例。
#
# 白色：原始掩膜；绿色：内部区域；橙色：外扩边界。
# 右侧贡献只解释 LR，不能当作 TabICLv2 的解释。

for dataset in ["BrEaST", "BUS-UCLM"]:
    for page in [1, 2]:
        stem = f"{dataset}_cases_{page}"

        for suffix in [".png", ".pdf"]:
            (OUT / (stem + suffix)).write_bytes(
                read_bytes(stem + suffix)
            )

        print("\n病例图：", stem)
        with Image.open(
            io.BytesIO(read_bytes(stem + ".png"))
        ) as im:
            preview = im.copy()
            preview.thumbnail((1400, 1600))
            display(preview)

# ---------- 5. 保存说明、打包 ----------
notes = """
Frozen base_LR interpretation; no retraining or threshold changes.
Global ranking uses equal total weight per patient cluster.
Heatmap rows are cases/images, not pixels.
Red/blue indicates positive/negative calibrated-logit contribution
relative to the training-mean reference, not causal effects.
Case region contours are not saliency maps.
Case plots explain LR only; TabICLv2 probabilities are comparisons.
Group sizes differ, so group absolute mass is not biological importance.
BUS-UCLM contains 207 images from 33 patient clusters.
"""
(OUT / "figure_notes.txt").write_text(notes, encoding="utf-8")

archive = shutil.make_archive(str(OUT), "zip", OUT)

print("\n完成。没有重新训练模型，也没有修改任何阈值。")
print("输出目录：", OUT)
print("请下载备份：", archive)
display(FileLink(archive))

# ==================== 代码结束 ====================

In [ ]:
%pip install -q scikit-learn==1.6.1

In [ ]:
# ==================== START ====================
from pathlib import Path
from datetime import datetime
import io
import json
import hashlib
import zipfile
import shutil

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from scipy.special import expit, logit
from scipy.stats import spearmanr
from IPython.display import display, FileLink

# 自动寻找；存在多个版本时填写明确路径。
MODEL_PATH = ""
CONTOUR_PATH = ""

ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working") / (
    "bus_v12_explanation_stability_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
)
OUT.mkdir(parents=True, exist_ok=True)

N_BOOT = 2000
SEED = 20261014

# 校准后 log-odds 贡献的幅度阈值：
# 是本次描述性分析约定，不是临床阈值。
SIGN_EPS = 0.10

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
})

def discover(explicit, stem):
    if explicit.strip():
        p = Path(explicit.strip())
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    matches = sorted({
        p for p in ROOT.rglob(stem + "*")
        if p.is_dir() or p.suffix.lower() == ".zip"
    })

    # 同一路径下同时有 ZIP 和其解压目录时优先 ZIP。
    zipped = [p for p in matches if p.is_file()]
    if len(zipped) == 1:
        return zipped[0]
    if len(matches) == 1:
        return matches[0]

    print("候选：")
    for p in matches:
        print(p)
    raise RuntimeError(
        "未找到唯一输入，请在顶部填写明确路径： " + stem
    )

model_source = discover(
    MODEL_PATH,
    "bus_v6_external_training_20261013_121259_911015"
)
contour_source = discover(
    CONTOUR_PATH,
    "bus_uclm_contour_robustness_20261013_135101_583248"
)

def read_bytes(source, name):
    if source.is_dir():
        p = source / name
        if p.is_file():
            return p.read_bytes()
        hits = [
            x for x in source.rglob(Path(name).name)
            if x.as_posix().endswith("/" + name)
        ]
        if len(hits) != 1:
            raise RuntimeError(f"{name}: 找到 {len(hits)} 个")
        return hits[0].read_bytes()

    with zipfile.ZipFile(source) as z:
        hits = [
            n for n in z.namelist()
            if n == name or n.endswith("/" + name)
        ]
        if len(hits) != 1:
            raise RuntimeError(f"{name}: 找到 {len(hits)} 个")
        return z.read(hits[0])

def read_table(source, name):
    return pd.read_csv(
        io.BytesIO(read_bytes(source, name)),
        dtype={"image_id": str, "cluster_id": str},
        float_precision="round_trip",
    )

print("模型：", model_source)
print("扰动结果：", contour_source)

# ---------- 1. 读取并核对模型 ----------
model_bytes = read_bytes(
    model_source, "models/base_LR.joblib"
)
model_hash = hashlib.sha256(model_bytes).hexdigest()
hashes = json.loads(
    read_bytes(model_source, "model_hashes.json")
)
if model_hash != hashes["base_LR.joblib"]:
    raise ValueError("模型哈希不一致。")

# 只加载你自己生成并备份的模型包。
bundle = joblib.load(io.BytesIO(model_bytes))
pipe = bundle["models"]["LR"]
calibrator = bundle["calibrator"]
columns = bundle["feature_columns"]

a = float(calibrator.coef_[0, 0])
c = float(calibrator.intercept_[0])
baseline = a * float(pipe[-1].intercept_[0]) + c

features = read_table(
    contour_source, "perturbed_features.csv"
)
predictions = read_table(
    contour_source, "all_predictions.csv"
)
cohort = read_table(
    contour_source, "cohort_frozen.csv"
)

modes = ["original", "erode1", "dilate1"]
if not set(modes).issubset(set(features["mode"])):
    raise ValueError("缺少原始、内缩或外扩条件。")

if features.duplicated(["image_id", "mode"]).any():
    raise ValueError("特征表存在重复的图像和条件。")

groups = {
    "Shape": [
        j for j, x in enumerate(columns)
        if x.startswith("shape__")
    ],
    "Core": [
        j for j, x in enumerate(columns)
        if x.startswith("core__")
    ],
    "Boundary": [
        j for j, x in enumerate(columns)
        if x.startswith("rim__")
    ],
    "Boundary-core contrast": [
        j for j, x in enumerate(columns)
        if x.startswith("rim_core__")
    ],
}
if sorted(sum(groups.values(), [])) != list(range(len(columns))):
    raise ValueError("存在未归组或重复归组的特征。")

protocol = {
    "model": "Frozen base_LR",
    "model_sha256": model_hash,
    "conditions": modes,
    "perturbation": (
        "One 4-neighbour erosion/dilation iteration "
        "after frozen resizing; pixels, not millimetres"
    ),
    "refit": False,
    "threshold_changed": False,
    "contribution": "a * beta_j * standardized_feature_j",
    "sign_epsilon": SIGN_EPS,
    "bootstrap": (
        "2000 patient-cluster resamples; "
        "equal weight per cluster; percentile intervals"
    ),
    "scope": (
        "Descriptive contour stress test, "
        "not inter-reader or independent external validation"
    ),
}
(OUT / "protocol.json").write_text(
    json.dumps(protocol, indent=2), encoding="utf-8"
)

# ---------- 2. 对齐共同可评估队列 ----------
sets = [
    set(features.loc[features["mode"].eq(m), "image_id"])
    for m in modes
]
ids = sorted(set.intersection(*sets))
if not ids:
    raise ValueError("没有共同可评估图像。")

coverage = pd.DataFrame([
    {
        "mode": m,
        "available_images": len(s),
        "common_images": len(ids),
        "excluded_from_common": len(s - set(ids)),
    }
    for m, s in zip(modes, sets)
])
coverage.to_csv(OUT / "coverage.csv", index=False)

excluded = cohort[
    ~cohort["image_id"].isin(ids)
].copy()
excluded.to_csv(
    OUT / "excluded_from_common.csv", index=False
)

# ---------- 3. 计算贡献，核对原预测 ----------
C = {}
P = {}
REF = {}
checks = []

for mode in modes:
    g = (
        features[features["mode"].eq(mode)]
        .set_index("image_id")
        .loc[ids]
    )
    ref = predictions[
        predictions["mode"].eq(mode)
        & predictions["configuration"].eq("base_LR")
    ].set_index("image_id")

    if not ref.index.is_unique:
        raise ValueError("预测表存在重复图像。")
    ref = ref.loc[ids]

    X = g[columns]
    Z = pipe[:-1].transform(X)
    linear = pipe[-1].decision_function(Z)

    contributions = Z * pipe[-1].coef_[0] * a
    raw = pipe.predict_proba(X)[:, 1]

    correction = a * (
        logit(np.clip(raw, 1e-6, 1 - 1e-6)) - linear
    )
    probability = expit(
        baseline + contributions.sum(axis=1) + correction
    )

    error = float(np.max(
        np.abs(probability - ref["probability"].to_numpy())
    ))
    if error >= 1e-8:
        raise ValueError(
            f"{mode}预测未还原，误差={error}；停止分析。"
        )

    if ref["cluster_id"].isna().any():
        raise ValueError("患者簇 ID 缺失。")

    C[mode] = contributions
    P[mode] = probability
    REF[mode] = ref

    table = pd.DataFrame(
        contributions, index=ids, columns=columns
    )
    table.index.name = "image_id"
    table["clipping_correction"] = correction
    table.to_csv(OUT / f"{mode}_contributions.csv")

    checks.append({
        "mode": mode,
        "n": len(ids),
        "max_abs_prediction_error": error,
        "raw_clipped": int(
            ((raw < 1e-6) | (raw > 1 - 1e-6)).sum()
        ),
    })

check_table = pd.DataFrame(checks)
check_table.to_csv(
    OUT / "prediction_reproduction.csv", index=False
)
display(check_table)

original = C["original"]
reference = REF["original"]

# ---------- 4. 逐图像的解释变化 ----------
records = []

for mode in ["erode1", "dilate1"]:
    other = C[mode]

    if not np.array_equal(
        reference["cluster_id"], REF[mode]["cluster_id"]
    ):
        raise ValueError("扰动条件的患者簇不一致。")
    if not np.array_equal(
        reference["label"], REF[mode]["label"]
    ):
        raise ValueError("扰动条件的标签不一致。")
    if not np.array_equal(
        reference["threshold"], REF[mode]["threshold"]
    ):
        raise ValueError("扰动条件使用了不同阈值。")

    for i, image_id in enumerate(ids):
        x = original[i]
        y = other[i]

        # 相同绝对贡献时按固定列顺序打破并列。
        top_x = set(
            np.argsort(-np.abs(x), kind="stable")[:5]
        )
        top_y = set(
            np.argsort(-np.abs(y), kind="stable")[:5]
        )

        if np.std(np.abs(x)) == 0 or np.std(np.abs(y)) == 0:
            rho = np.nan
        else:
            rho = float(spearmanr(
                np.abs(x), np.abs(y)
            ).statistic)

        active = np.abs(x) >= SIGN_EPS
        reverse = (
            active
            & (np.abs(y) >= SIGN_EPS)
            & (np.sign(x) != np.sign(y))
        )

        # 原始贡献足够大，但扰动后接近零，单独统计。
        fade = active & (np.abs(y) < SIGN_EPS)

        row = {
            "image_id": image_id,
            "cluster_id": reference.iloc[i]["cluster_id"],
            "mode": mode,
            "label": int(reference.iloc[i]["label"]),
            "rank_spearman": rho,
            "top5_overlap": len(top_x & top_y) / 5,
            "active_features": int(active.sum()),
            "substantial_sign_flip": (
                reverse.sum() / active.sum()
                if active.any() else np.nan
            ),
            "active_to_near_zero": (
                fade.sum() / active.sum()
                if active.any() else np.nan
            ),
            "mean_abs_contribution_change": float(
                np.mean(np.abs(y - x))
            ),
            "abs_probability_change": float(
                abs(P[mode][i] - P["original"][i])
            ),
            # 复用已保存的原判定，避免另行改变边界规则。
            "prediction_flip": int(
                REF[mode].iloc[i]["predicted_label"]
                != reference.iloc[i]["predicted_label"]
            ),
        }

        for name, indices in groups.items():
            row[name + "_abs_group_change"] = float(abs(
                y[indices].sum() - x[indices].sum()
            ))

        records.append(row)

paired = pd.DataFrame(records)
paired.to_csv(
    OUT / "paired_explanation_changes.csv", index=False
)

# ---------- 5. 患者簇等权汇总及 bootstrap ----------
# 先计算每个患者簇内的图像均值，
# 再对患者簇进行有放回抽样。
metrics = [
    "rank_spearman",
    "top5_overlap",
    "substantial_sign_flip",
    "active_to_near_zero",
    "mean_abs_contribution_change",
    "abs_probability_change",
    "prediction_flip",
] + [x + "_abs_group_change" for x in groups]

rng = np.random.default_rng(SEED)
summary = []

for mode in ["erode1", "dilate1"]:
    g = paired[paired["mode"].eq(mode)]
    cluster_means = g.groupby("cluster_id")[metrics].mean()
    cluster_means.to_csv(
        OUT / f"{mode}_cluster_means.csv"
    )

    for metric in metrics:
        values = cluster_means[metric].to_numpy()
        values = values[np.isfinite(values)]

        if len(values) == 0:
            estimate = low = high = np.nan
        else:
            resampled = rng.integers(
                0, len(values),
                size=(N_BOOT, len(values)),
            )
            boot = values[resampled].mean(axis=1)
            estimate = float(values.mean())
            low, high = np.quantile(boot, [0.025, 0.975])

        summary.append({
            "mode": mode,
            "metric": metric,
            "cluster_equal_mean": estimate,
            "ci_low": low,
            "ci_high": high,
            "valid_clusters": len(values),
            "valid_images": int(g[metric].notna().sum()),
        })

summary = pd.DataFrame(summary)
summary.to_csv(
    OUT / "cluster_bootstrap_summary.csv", index=False
)

print("\n共同可评估图像：", len(ids))
print("患者簇：", reference["cluster_id"].nunique())
display(coverage)
display(summary)

# ---------- 6. 图：解释稳定性 ----------
def save_plot(fig, filename):
    fig.savefig(
        OUT / (filename + ".png"),
        dpi=300, bbox_inches="tight",
    )
    fig.savefig(
        OUT / (filename + ".pdf"),
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

titles = {
    "rank_spearman": "Absolute-contribution rank correlation",
    "top5_overlap": "Top-5 feature overlap",
    "substantial_sign_flip": "Substantial contribution sign reversal",
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (metric, title) in zip(axes, titles.items()):
    s = (
        summary[summary["metric"].eq(metric)]
        .set_index("mode")
        .loc[["erode1", "dilate1"]]
    )

    for i, (_, row) in enumerate(s.iterrows()):
        ax.vlines(i, row.ci_low, row.ci_high, color="#287da5")
        ax.scatter(i, row.cluster_equal_mean,
                   color="#287da5", s=45, zorder=3)

    ax.set_xticks([0, 1], ["Erosion", "Dilation"])
    ax.set_xlim(-0.5, 1.5)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("Cluster-equal mean and 95% bootstrap CI")
    ax.grid(axis="y", alpha=0.2)

fig.tight_layout()
save_plot(fig, "Figure_explanation_stability")

# ---------- 7. 图：预测变化与解释变化 ----------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, mode in zip(axes, ["erode1", "dilate1"]):
    g = paired[paired["mode"].eq(mode)]

    for flag, color, label in [
        (0, "#287da5", "Classification unchanged"),
        (1, "#c85c46", "Classification changed"),
    ]:
        sub = g[g["prediction_flip"].eq(flag)]
        ax.scatter(
            sub["abs_probability_change"],
            sub["mean_abs_contribution_change"],
            color=color, alpha=0.65, s=22, label=label,
        )

    ax.set_title(mode)
    ax.set_xlabel("Absolute probability change")
    ax.set_ylabel("Mean absolute contribution change")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.15)

fig.suptitle(
    "Image-level descriptive plot; images are clustered within patients",
    fontsize=10,
)
fig.tight_layout()
save_plot(fig, "Figure_prediction_vs_explanation")

# ---------- 8. 图：各区域贡献变化 ----------
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(groups))
width = 0.34

for offset, mode, color in [
    (-width/2, "erode1", "#287da5"),
    (width/2, "dilate1", "#c85c46"),
]:
    values = []
    for group in groups:
        row = summary[
            summary["mode"].eq(mode)
            & summary["metric"].eq(group + "_abs_group_change")
        ].iloc[0]
        values.append(row["cluster_equal_mean"])

    ax.bar(x + offset, values, width,
           label=mode, color=color)

ax.set_xticks(x, list(groups), rotation=15)
ax.set_ylabel("Mean absolute change in signed group contribution")
ax.set_title(
    "Cluster-equal summary; groups contain different feature counts"
)
ax.legend()
fig.tight_layout()
save_plot(fig, "Figure_group_contribution_changes")

# ---------- 9. 保存说明与代码 ----------
notes = """
This analysis explains the frozen base_LR only.
No diagnostic model was fitted or recalibrated.
Original decision thresholds and saved classifications were retained.
Perturbations are pixel-based after resizing, not physical millimetres.
This is a descriptive stress test, not inter-reader agreement.

Rank correlation uses absolute individual contributions.
Top-5 overlap is the intersection size divided by five.
Substantial sign reversal requires opposite signs and absolute
contributions >= 0.10 in both conditions. Its denominator is the
number of original contributions with absolute value >= 0.10.
Fading to near-zero is reported separately.
The 0.10 cutoff is descriptive, not clinically validated.

Bootstrap intervals summarize equal-weight patient clusters.
They are metric-specific descriptive intervals, not simultaneous
confidence intervals or formal clinical acceptance criteria.
Group contribution sums may cancel, and group sizes differ.
Prediction reconstruction does not itself prove explanation stability.
The source ensemble sensitivity-threshold audit is not addressed here.
"""
(OUT / "README.txt").write_text(notes, encoding="utf-8")

try:
    code = "\n\n".join(
        get_ipython().history_manager.input_hist_raw[1:]
    )
    (OUT / "executed_code.py").write_text(
        code, encoding="utf-8"
    )
except Exception:
    pass

archive = shutil.make_archive(str(OUT), "zip", OUT)
print("\n完成，请下载结果包：", archive)
display(FileLink(archive))

# ==================== END OF CODE ====================

In [ ]:
%pip install -q scikit-learn==1.6.1

In [ ]:
# ==================== START ====================
from pathlib import Path
from datetime import datetime
import io
import json
import zipfile
import hashlib
import shutil
import platform

import numpy as np
import pandas as pd
import scipy
import sklearn
import joblib

from scipy.special import logit
from IPython.display import display, FileLink

# 自动发现失败或存在多个版本时填写明确路径。
MODEL_PATH = ""
FEATURE_PATH = (
    "/kaggle/input/datasets/kokomixueyue/bus-v16/"
    "bus_v6_features_20261013_120420_94529"
)

ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working") / (
    "bus_v13_threshold_audit_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
)
OUT.mkdir(parents=True, exist_ok=True)


def discover(explicit, stem):
    if explicit.strip():
        p = Path(explicit.strip())
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    candidates = sorted({
        p for p in ROOT.rglob(stem + "*")
        if p.is_dir() or p.suffix.lower() == ".zip"
    })
    archives = [p for p in candidates if p.is_file()]

    if len(archives) == 1:
        return archives[0]
    if len(candidates) == 1:
        return candidates[0]

    print("候选位置：")
    for p in candidates:
        print(p)
    raise RuntimeError(
        "请在代码顶部填写明确路径：" + stem
    )


def read_bytes(source, name):
    if source.is_dir():
        direct = source / name
        if direct.is_file():
            return direct.read_bytes()

        matches = [
            p for p in source.rglob(Path(name).name)
            if p.as_posix().endswith("/" + name)
        ]
        if len(matches) != 1:
            raise RuntimeError(
                f"{name} 匹配到 {len(matches)} 个文件。"
            )
        return matches[0].read_bytes()

    with zipfile.ZipFile(source) as z:
        matches = [
            n for n in z.namelist()
            if n == name or n.endswith("/" + name)
        ]
        if len(matches) != 1:
            raise RuntimeError(
                f"{name} 匹配到 {len(matches)} 个文件。"
            )
        return z.read(matches[0])


def table(source, name):
    return pd.read_csv(
        io.BytesIO(read_bytes(source, name)),
        dtype={"uid": str, "patient_key": str},
        float_precision="round_trip",
    )


model_source = discover(
    MODEL_PATH,
    "bus_v6_external_training_20261013_121259_911015"
)
feature_source = discover(
    FEATURE_PATH,
    "bus_v6_features_20261013_120420_94529"
)

print("模型包：", model_source)
print("特征包：", feature_source)

# ---------- 1. 保存当前环境 ----------
environment = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
}
(OUT / "environment.json").write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)
display(pd.DataFrame([environment]))

# ---------- 2. 对比两种 CSV 读取方式 ----------
saved_bytes = read_bytes(
    model_source, "source_threshold_predictions.csv"
)

saved = pd.read_csv(
    io.BytesIO(saved_bytes),
    dtype={"patient_key": str},
    float_precision="round_trip",
)
default_read = pd.read_csv(
    io.BytesIO(saved_bytes),
    dtype={"patient_key": str},
)

keys = ["feature_set", "model", "patient_key"]

if saved.duplicated(keys).any():
    raise ValueError("保存预测表存在重复记录。")
if default_read.duplicated(keys).any():
    raise ValueError("默认读取表存在重复记录。")

saved = saved.set_index(keys).sort_index()
default_read = default_read.set_index(keys).loc[saved.index]

threshold_audit = table(
    model_source, "threshold_audit.csv"
)
hashes = json.loads(
    read_bytes(model_source, "model_hashes.json")
)
training_protocol = json.loads(
    read_bytes(model_source, "protocol.json")
)

# ---------- 3. 核对输入特征文件哈希 ----------
expected_inputs = training_protocol.get("input_hashes", {})
for name in ["features.csv", "cohort.csv"]:
    content = read_bytes(feature_source, name)
    actual = hashlib.sha256(content).hexdigest()
    expected = expected_inputs.get(name)

    if expected is not None and actual != expected:
        raise ValueError(
            f"{name} 与原训练使用的输入哈希不一致，停止。"
        )

cohort = table(feature_source, "cohort.csv")
features = table(feature_source, "features.csv")

if not cohort["uid"].is_unique or not features["uid"].is_unique:
    raise ValueError("清单或特征 UID 重复。")
if set(cohort["uid"]) != set(features["uid"]):
    raise ValueError("清单与特征 UID 不匹配。")

frame = cohort.merge(
    features, on="uid", validate="one_to_one"
)
frame = frame[frame["dataset"].eq("BUS-BRA")].copy()

if not frame.groupby("patient_key")["label"].nunique().eq(1).all():
    raise ValueError("同一患者存在不同标签。")

source_labels = frame.groupby("patient_key")["label"].first()

# ---------- 4. 原模型重新推理 ----------
all_details = []
summaries = []

for feature_set, model_name in (
    saved.reset_index()[["feature_set", "model"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
):
    filename = f"{feature_set}_{model_name}.joblib"
    content = read_bytes(model_source, "models/" + filename)

    if hashlib.sha256(content).hexdigest() != hashes[filename]:
        raise ValueError(filename + " 模型哈希不一致。")

    # 只加载你自己生成、备份的模型包。
    bundle = joblib.load(io.BytesIO(content))
    threshold = float(bundle["threshold"])
    columns = bundle["feature_columns"]

    old = saved.loc[(feature_set, model_name)].copy()
    old_default = default_read.loc[(feature_set, model_name)].copy()
    ids = old.index.tolist()

    if not set(ids).issubset(set(frame["patient_key"])):
        raise ValueError("特征表缺少阈值开发集患者。")

    gate = frame[frame["patient_key"].isin(ids)].copy()

    if not np.array_equal(
        source_labels.loc[ids].to_numpy(),
        old["label"].to_numpy()
    ):
        raise ValueError("患者标签与原预测表不一致。")

    if not np.all(old["threshold"].to_numpy() == threshold):
        raise ValueError(
            "高精度读取的保存阈值与模型内阈值不一致。"
        )

    # 严格复用原顺序：
    # 每个模型先按患者平均图像概率，
    # 然后对 LR/ET/HGB 患者概率取均值，再校准。
    member_names = (
        ["LR"] if model_name == "LR"
        else ["LR", "ET", "HGB"]
    )
    patient_probabilities = []

    for member_name in member_names:
        estimator = bundle["models"][member_name]
        image_p = estimator.predict_proba(
            gate[columns]
        )[:, 1]

        patient_p = (
            pd.Series(
                image_p,
                index=gate["patient_key"].to_numpy()
            )
            .groupby(level=0)
            .mean()
            .sort_index()
        )
        patient_probabilities.append(
            patient_p.rename(member_name)
        )

    raw = pd.concat(patient_probabilities, axis=1).mean(axis=1)
    calibrated = bundle["calibrator"].predict_proba(
        logit(
            np.clip(raw.to_numpy(), 1e-6, 1 - 1e-6)
        ).reshape(-1, 1)
    )[:, 1]

    new_p = pd.Series(
        calibrated, index=raw.index
    ).loc[ids].to_numpy()

    old_p = old["p"].to_numpy(dtype=float)
    old_t = old["threshold"].to_numpy(dtype=float)
    default_p = old_default["p"].to_numpy(dtype=float)
    default_t = old_default["threshold"].to_numpy(dtype=float)

    y = old["label"].to_numpy(dtype=int)
    stored_pred = old["prediction"].to_numpy(dtype=int)

    # 正式规则仍是 >=；不使用容差修正判定。
    roundtrip_pred = (old_p >= old_t).astype(int)
    default_pred = (default_p >= default_t).astype(int)
    new_pred = (new_p >= threshold).astype(int)

    if not set(y).issubset({0, 1}):
        raise ValueError("标签不是0/1。")

    audit_row = threshold_audit[
        threshold_audit["feature_set"].eq(feature_set)
        & threshold_audit["model"].eq(model_name)
    ]
    if len(audit_row) != 1:
        raise ValueError("原阈值审计表匹配异常。")

    detail = pd.DataFrame({
        "feature_set": feature_set,
        "model": model_name,
        "patient_key": ids,
        "label": y,
        "saved_p_roundtrip": old_p,
        "saved_threshold_roundtrip": old_t,
        "saved_prediction": stored_pred,
        "roundtrip_prediction": roundtrip_pred,
        "default_csv_p": default_p,
        "default_csv_threshold": default_t,
        "default_csv_prediction": default_pred,
        "recomputed_p": new_p,
        "frozen_threshold": threshold,
        "recomputed_prediction": new_pred,
        "saved_margin": old_p - threshold,
        "recomputed_margin": new_p - threshold,
        "probability_difference": new_p - old_p,
        "csv_parser_changes_decision":
            default_pred != roundtrip_pred,
        "recompute_changes_saved_decision":
            new_pred != stored_pred,
        # 仅标记临界样本，不用于改判。
        "near_threshold_1e12":
            np.abs(old_p - threshold) <= 1e-12,
        "saved_exact_tie": old_p == threshold,
    })

    # 十六进制浮点表示，避免显示精度掩盖微小差异。
    detail["saved_p_hex"] = [float(x).hex() for x in old_p]
    detail["recomputed_p_hex"] = [float(x).hex() for x in new_p]
    detail["threshold_hex"] = threshold.hex()
    all_details.append(detail)

    for method, predicted in [
        ("original_saved_decision", stored_pred),
        ("csv_roundtrip_ge", roundtrip_pred),
        ("csv_default_ge", default_pred),
        ("current_runtime_ge", new_pred),
    ]:
        tp = int(((y == 1) & (predicted == 1)).sum())
        fn = int(((y == 1) & (predicted == 0)).sum())
        tn = int(((y == 0) & (predicted == 0)).sum())
        fp = int(((y == 0) & (predicted == 1)).sum())

        summaries.append({
            "feature_set": feature_set,
            "model": model_name,
            "method": method,
            "patients": len(y),
            "malignant": int((y == 1).sum()),
            "TP": tp,
            "FN": fn,
            "TN": tn,
            "FP": fp,
            "sensitivity": tp / (tp + fn),
            "specificity": tn / (tn + fp),
            "frozen_threshold": threshold,
            "original_reported_sensitivity": float(
                audit_row.iloc[0]["development_sensitivity"]
            ),
            "max_abs_recomputed_probability_difference":
                float(np.max(np.abs(new_p - old_p))),
            "changed_vs_original_saved":
                int((predicted != stored_pred).sum()),
        })

    print("完成核查：", feature_set, model_name)

# ---------- 5. 保存所有结果 ----------
details = pd.concat(all_details, ignore_index=True)
summary = pd.DataFrame(summaries)

# 保留17位有效数字，减少再次导出引入的信息损失。
details.to_csv(
    OUT / "all_patient_audit.csv",
    index=False, float_format="%.17g",
)
summary.to_csv(
    OUT / "threshold_audit_summary.csv",
    index=False, float_format="%.17g",
)

flag = (
    details["near_threshold_1e12"]
    | details["csv_parser_changes_decision"]
    | details["recompute_changes_saved_decision"]
    | (
        details["saved_prediction"]
        != details["roundtrip_prediction"]
    )
)
boundary = details[flag].copy()
boundary.to_csv(
    OUT / "boundary_and_disagreement_cases.csv",
    index=False, float_format="%.17g",
)

# 保留原始CSV字节，便于追溯，不覆盖输入文件。
(OUT / "original_source_threshold_predictions.csv").write_bytes(
    saved_bytes
)

notes = """
This is a reproducibility audit, not a new performance experiment.
No model was fitted; no threshold was changed.
Original saved decisions, round-trip CSV parsing, default CSV parsing,
and current-runtime inference are reported separately.

The >= rule is retained. A 1e-12 proximity flag is diagnostic only;
it is not used to alter decisions or rescue sensitivity.

CSV parsing differences and current-runtime differences are distinct.
A tiny probability difference at a threshold can change a binary
decision. Current reproduction cannot prove the exact cause of a
historical V8 run unless its individual source predictions and
runtime details were retained.

Threshold-subset sensitivity is developmental, not independent
test performance. Original external results are not overwritten.
"""
(OUT / "README.txt").write_text(notes, encoding="utf-8")

try:
    code = "\n\n".join(
        get_ipython().history_manager.input_hist_raw[1:]
    )
    (OUT / "executed_code.py").write_text(
        code, encoding="utf-8"
    )
except Exception:
    pass

print("\n各读取/运行方式下的开发集结果：")
display(summary)

print("\n临界或判定不一致的患者：")
display(boundary[[
    "feature_set", "model", "patient_key", "label",
    "saved_p_roundtrip", "frozen_threshold",
    "saved_prediction", "roundtrip_prediction",
    "default_csv_prediction", "recomputed_prediction",
    "probability_difference",
]])

archive = shutil.make_archive(str(OUT), "zip", OUT)
print("\n核查完成，没有修改任何模型、阈值或外部结果。")
print("请下载：", archive)
display(FileLink(archive))

# ==================== END ====================